# Lesson 12 · Momentum and collisions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/12_momentum_and_collisions.ipynb)

**For:** A-level Physics (for example AQA 3.4.1.6, OCR A 3.5, Edexcel Topics 2 and 6) and IB Physics (A.2 forces and momentum). Check your own specification for the exact wording.

**Before this lesson:** filled in by `scripts/lesson_path.py`.

**Leads to:** filled in by `scripts/lesson_path.py`.

**You will be able to:**

1. calculate momentum $p = mv$ and use Newton's second law in the form $F = \Delta p/\Delta t$
2. find an impulse $F\Delta t$ from a force and from the area under a force–time graph
3. explain why momentum is conserved in every collision and explosion of an isolated system
4. tell elastic from inelastic collisions using kinetic energy
5. solve collisions and explosions in one dimension, and glancing collisions in two

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Install the course API in a fresh Colab runtime or upgrade an older installation.
import importlib.util
import subprocess
import sys
from importlib.metadata import version

if importlib.util.find_spec("qplay") is None or int(version("quantum-playground").split(".")[0]) < 1:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "quantum-playground>=1,<2"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import mechanics

## 1. Momentum and impulse

The **momentum** of an object is its mass times its velocity,

$$p = mv \qquad (\text{unit: kg m s}^{-1}\text{, or N s}).$$

Momentum is a vector: it points the same way as the velocity, so a ball moving left has negative momentum if right is positive.

Newton's second law is really a statement about momentum: the resultant force equals the **rate of change of momentum**,

$$F = \frac{\Delta p}{\Delta t}.$$

For a constant mass this is the familiar $F = ma$. Rearranged, $F\Delta t = \Delta p$. The product $F\Delta t$ is the **impulse**. When the force varies, as it does whenever a bat hits a ball, the impulse is the **area under the force–time graph**.

**Predict (P1).** A 60 g egg falling at 5.0 m/s is stopped either by a hard floor, in 2 ms, or by a cushion, in 100 ms. Which one gives the egg the bigger impulse? Which one exerts the bigger force? Write your predictions down before you run the next cell.

In [ ]:
m, u = 0.060, 5.0                          # kg, m/s
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
for ax, stop_time, label in [(axes[0], 0.002, "hard floor"), (axes[1], 0.100, "cushion")]:
    t = np.linspace(0, stop_time, 400)
    peak = np.pi * m * u / (2 * stop_time)      # a half-sine pulse whose area is m*u
    force = peak * np.sin(np.pi * t / stop_time)
    impulse = np.trapezoid(force, t)            # area under the force-time graph
    ax.fill_between(1e3 * t, force, alpha=0.3)
    ax.plot(1e3 * t, force)
    ax.set(xlabel="t / ms", ylabel="force on the egg / N", title=f"{label}: shaded area = {impulse:.3f} N s")
    print(f"{label:>10}: change in momentum m*u = {m * u:.3f} N s, impulse = {impulse:.3f} N s, "
          f"average force {impulse / stop_time:6.1f} N, peak force {force.max():6.1f} N")
fig.suptitle("Stopping a 60 g egg moving at 5.0 m/s (note the different scales)");

**Explain (P1).** Why are the two impulses equal? Why is one force so much larger than the other?

*Try it:* this is how crumple zones, airbags, cycle helmets and the bent knees of a landing gymnast work. Change the cushion's stopping time to 0.5 s.

## 2. Conservation of momentum

When two objects collide, Newton's third law says each pushes on the other with an equal and opposite force, for exactly the same time. So their impulses are equal and opposite: whatever momentum one gains, the other loses. In an **isolated system**, one with no resultant external force,

$$\text{total momentum before} = \text{total momentum after}.$$

This is true in **every** collision and explosion. Kinetic energy is a different story:

- in an **elastic** collision the total kinetic energy is also conserved
- in an **inelastic** collision some kinetic energy becomes thermal energy, sound and permanent deformation
- in a **perfectly inelastic** collision the objects stick together, and the largest possible amount of kinetic energy is lost

`mechanics.collision(m1, m2, u1, u2, restitution=e)` returns the velocities after an impact. The restitution $e$ sets how bouncy it is: 1 for elastic, 0 for objects that stick together.

**Predict (P2).** A 1.0 kg trolley moving at 3.0 m/s hits an identical trolley at rest, and the collision is elastic. What are the two velocities afterwards?

In [ ]:
def report(m1, m2, u1, u2, e):
    (v1,), (v2,) = mechanics.collision(m1, m2, [u1], [u2], restitution=e)
    p_before, p_after = m1 * u1 + m2 * u2, m1 * v1 + m2 * v2
    ke_before = 0.5 * m1 * u1**2 + 0.5 * m2 * u2**2
    ke_after = 0.5 * m1 * v1**2 + 0.5 * m2 * v2**2
    print(f"e = {e:3.1f}: v1 = {v1:+5.2f} m/s, v2 = {v2:+5.2f} m/s | momentum {p_before:.2f} -> {p_after:.2f} N s"
          f" | kinetic energy {ke_before:.2f} -> {ke_after:.2f} J")

report(1.0, 1.0, 3.0, 0.0, e=1.0)

**Explain (P2).** Use the two printed totals to explain why no other pair of velocities would do.

Now make the collision less and less bouncy. Momentum stays the same every time; watch the kinetic energy.

In [ ]:
for e in (1.0, 0.5, 0.0):
    report(2.0, 1.0, 3.0, 0.0, e)

At $e = 0$ the trolleys move off together at $2.0$ m/s and a third of the kinetic energy has gone.

## 3. Explosions and recoil

In an explosion the objects start at rest, so the total momentum is zero before **and after**: the pieces fly apart with equal and opposite momenta. A 4.0 kg rifle that fires a 20 g bullet at 400 m/s recoils at

$$v = -\frac{0.020 \times 400}{4.0} = -2.0\ \mathrm{m\,s^{-1}}.$$

The kinetic energy is not conserved here either: it comes from the chemical energy of the propellant. The bullet ends up with $\tfrac12 \times 0.020 \times 400^2 = 1600$ J, the rifle with only $\tfrac12 \times 4.0 \times 2.0^2 = 8$ J. Equal momenta, very unequal energies, because $E_k = p^2/2m$.

## 4. Collisions in two dimensions

When two smooth pucks or snooker balls collide, they push on each other only along the **line of centres** at the moment of contact. Momentum is conserved in each direction separately, so resolve the velocities into components along and across that line.

**Predict (P3).** A puck sliding at 3.0 m/s strikes an identical puck at rest off-centre, so the struck puck moves off at an angle. The collision is elastic. What is the angle between the two pucks' paths afterwards? Does it depend on how off-centre the hit is?

In [ ]:
u1 = np.array([3.0, 0.0])
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.annotate("", xy=(0, 0), xytext=(-3, 0), arrowprops=dict(arrowstyle="->", color="0.4", lw=2))
ax.text(-1.5, 0.15, "before: 3.0 m/s", ha="center", color="0.4")
for i, angle in enumerate([20, 40, 60]):
    normal = [np.cos(np.radians(angle)), np.sin(np.radians(angle))]     # line of centres at impact
    v1, v2 = mechanics.collision(1.0, 1.0, u1, [0.0, 0.0], normal=normal, restitution=1.0)
    between = np.degrees(np.arccos(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2))))
    ax.annotate("", xy=v1, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=f"C{i}", lw=1.5, ls="--"))
    ax.annotate("", xy=v2, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=f"C{i}", lw=1.5))
    ax.plot([], [], color=f"C{i}", label=f"line of centres at {angle} degrees")
    print(f"line of centres at {angle} degrees: angle between the paths {between:.1f} degrees; "
          f"momentum after {v1 + v2} N s; kinetic energy {0.5 * (v1 @ v1 + v2 @ v2):.2f} J")
ax.set(xlim=(-3.3, 3.3), ylim=(-2.2, 3.0), aspect="equal", xlabel="x velocity / m s$^{-1}$", ylabel="y velocity / m s$^{-1}$")
ax.set_title("Velocities after impact: struck puck solid, incoming puck dashed")
ax.legend(fontsize=8, loc="upper left");

**Explain (P3).** Write momentum conservation as a vector triangle, $\mathbf u_1 = \mathbf v_1 + \mathbf v_2$, and kinetic-energy conservation as $u_1^2 = v_1^2 + v_2^2$ (the masses cancel). Which famous theorem do these two together describe?

**Common mistakes**

- Forgetting that momentum is a vector: give one direction a minus sign before you add.
- Assuming kinetic energy is conserved. It is only for elastic collisions; momentum always is.
- Using the mass of only one object after a collision in which they stick together.

---

## Exercises

Take right as positive. Give answers to 3 significant figures.

**Q1.** A 58 g tennis ball arrives at a racket at 20 m/s and leaves in the opposite direction at 30 m/s. The contact lasts 5.0 ms. Find the average force on the ball.

**Q2.** A 2.0 kg trolley moving at 3.0 m/s collides with a 1.0 kg trolley at rest, and they stick together. Find their speed afterwards.

**Q3.** Find the kinetic energy lost in the collision in Q2.

**Q4.** A 4.0 kg rifle fires a 20 g bullet at 400 m/s to the right. Find the velocity of the rifle (with its sign).

**Q5.** A 0.50 kg ball moving right at 4.0 m/s hits a 1.5 kg ball at rest head-on. The collision is elastic. Find the velocity of the 0.50 kg ball afterwards (with its sign).

**Q6 (simulation).** A puck moving at 3.0 m/s hits an identical puck at rest elastically. The struck puck moves off at 40° to the first puck's original direction. Find the struck puck's speed with `mechanics.collision`, then check it with the result of P3.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_answers = {
    "Q1": 0.058 * (30 - (-20)) / 0.005,
    "Q2": (2.0 * 3.0 + 1.0 * 0.0) / (2.0 + 1.0),
    "Q3": 0.5 * 2.0 * 3.0**2 - 0.5 * 3.0 * 2.0**2,
    "Q4": -0.020 * 400 / 4.0,
    "Q5": mechanics.collision(0.50, 1.5, [4.0], [0.0], restitution=1.0)[0][0],
    "Q6": np.linalg.norm(mechanics.collision(1.0, 1.0, [3.0, 0.0], [0.0, 0.0],
                                             normal=[np.cos(np.radians(40)), np.sin(np.radians(40))])[1]),
}

# Replace None with your answers, in SI units unless the question says otherwise.
mine = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** The change in velocity is $30 - (-20) = 50\ \mathrm{m\,s^{-1}}$, so $\Delta p = 0.058 \times 50 = 2.90$ N s and $F = \Delta p/\Delta t = 2.90/0.0050 = 580$ N.

**Q2.** Momentum before: $2.0 \times 3.0 = 6.0$ N s. Afterwards the combined 3.0 kg moves at $v = 6.0/3.0 = 2.00\ \mathrm{m\,s^{-1}}$.

**Q3.** Before: $\tfrac12 \times 2.0 \times 3.0^2 = 9.0$ J. After: $\tfrac12 \times 3.0 \times 2.0^2 = 6.0$ J. Lost: $3.00$ J, a third of the original.

**Q4.** Total momentum is zero before and after: $4.0\,v + 0.020 \times 400 = 0$, so $v = -2.00\ \mathrm{m\,s^{-1}}$ (2.0 m/s to the left).

**Q5.** Use both conservation laws: $0.50 \times 4.0 = 0.50\,v_1 + 1.5\,v_2$ and $\tfrac12 \times 0.50 \times 4.0^2 = \tfrac12 \times 0.50\,v_1^2 + \tfrac12 \times 1.5\,v_2^2$. Solving gives $v_1 = \dfrac{m_1 - m_2}{m_1 + m_2}u = -2.00\ \mathrm{m\,s^{-1}}$ (it bounces back) and $v_2 = 2.00\ \mathrm{m\,s^{-1}}$.

**Q6.** The struck puck moves along the line of centres. From P3 the two paths are at right angles, so the vector triangle is right-angled and the struck puck takes the component of $\mathbf u_1$ along its path: $v_2 = 3.0\cos 40^\circ = 2.30\ \mathrm{m\,s^{-1}}$. The other puck moves off at $3.0\sin 40^\circ = 1.93\ \mathrm{m\,s^{-1}}$.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}".rstrip("."))

### Predict and explain: answers

**P1.** The impulses are equal because the egg's momentum changes by the same amount, from $0.30$ N s to zero, whatever stops it: impulse equals change of momentum. The force is the impulse divided by the time, so stopping 50 times more slowly needs a force 50 times smaller (about 3 N on average instead of 150 N).

**P2.** The moving trolley stops dead and the other moves off at 3.0 m/s. These are the only velocities that keep both the momentum (3.0 N s) and the kinetic energy (4.5 J) the same, which an elastic collision must.

**P3.** Always $90^\circ$, however off-centre the hit. The vector triangle $\mathbf u_1 = \mathbf v_1 + \mathbf v_2$ has sides that also obey $u_1^2 = v_1^2 + v_2^2$: that is Pythagoras' theorem, so the angle between $\mathbf v_1$ and $\mathbf v_2$ is a right angle. Snooker players use this to predict where the cue ball goes.